# 01 Python + 监督学习基础

这一节用一个小型“材料性质”数据集完成完整回归流程：

`DataFrame → train/test split → scaling → model → MAE/RMSE/R² → parity plot`

重点是 workflow，而不是材料体系本身。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

rng = np.random.default_rng(42)
n = 300
df = pd.DataFrame({
    'density': rng.uniform(0.3, 1.5, n),
    'pore_diameter': rng.uniform(6, 35, n),
    'void_fraction': rng.uniform(0.25, 0.85, n),
    'N_fraction': rng.uniform(0.0, 0.20, n),
    'O_fraction': rng.uniform(0.0, 0.20, n),
})
df['target'] = 2.0*df['void_fraction'] + 0.08*df['pore_diameter'] + 3.0*df['N_fraction'] - 0.6*df['density'] + rng.normal(0,0.25,n)
df.head()

In [ ]:
X = df.drop(columns='target')
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)
models = {'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1.0)), 'RandomForest': RandomForestRegressor(n_estimators=300,random_state=42,n_jobs=-1)}
for name, model in models.items():
    model.fit(X_train,y_train)
    pred=model.predict(X_test)
    print(name, 'MAE=',mean_absolute_error(y_test,pred),'RMSE=',mean_squared_error(y_test,pred)**0.5,'R2=',r2_score(y_test,pred))

In [ ]:
model=models['RandomForest']; pred=model.predict(X_test)
plt.figure(figsize=(5,5)); plt.scatter(y_test,pred,alpha=0.7)
lims=[min(y_test.min(),pred.min()),max(y_test.max(),pred.max())]
plt.plot(lims,lims,'--'); plt.xlabel('True'); plt.ylabel('Predicted'); plt.title('Parity plot'); plt.show()

## 必做题

1. 将 `test_size` 改成 0.1、0.3，观察评价指标变化。
2. 去掉 `pore_diameter`，重新训练。
3. 比较 Ridge 与 Random Forest。
4. 写一句话解释：为什么训练集表现不能代表泛化能力？